В рамках работы использовался набор транзакционных данных о покупках, представленных в файле checks.csv. Изначально данные имели формат, в котором каждой покупке соответствовало несколько строк (по одной строке на товар), что является стандартным представлением транзакций.

Для повышения реалистичности анализа и усложнения структуры данных в исходный набор (test_set.csv, содержащий 9 транзакций) было дополнительно сгенерировано и добавлено 21 новая транзакция. В результате итоговый набор включает транзакции различной длины (разного количества товаров в одной покупке), что лучше отражает реальные покупательские сценарии.

Раньше набор транзакций содержал 9 записей, теперь -- 30.

In [10]:
import numpy as np
import pandas as pd
#from pandas.api.types import CategoricalDtype

class ARApriori():
    def __init__(self, filepath ,sep=';', items_col='Items', items_sep=',', header='infer', encoding="utf-8"):
        # Создаем DataFrame из файла csv
        self.dataset_in = pd.read_csv(filepath, sep=sep, header=header, names=None, encoding=encoding)
        # Настройки вывода DataFrame
        pd.set_option("display.max_rows", None, "display.max_columns", None)
        # Внутренний атрибут для хранения "формы" DataFrame
        self._ds_shape = self.dataset_in.shape
        self.dataset_final = pd.DataFrame({'Items': [], 'Support': []})  # DataFrame Для хранения конечного результата
        self.items_col = items_col      # Наименование колонки с наборами объектов/предметов
        self.items_sep = items_sep      # разделитель объектов/предметов в наборе

    def set_ds(self, dataset):
        self.dataset_in = dataset
        self._ds_shape = dataset.shape

    # вычисление поддержки (отношение количества записей, содержащих подмножество, к количесву всех записей в наборе данных)
    def _get_support(self, sup_cnt):
        return (sup_cnt*100)/self._ds_shape[0]

    # DataFrame - это изменяемый объект, т.о. при передаче в качестве аргумента ф-ии он не копируется!
    # Подсчет количества вхождений множества-кандидата в множества из набора данных
    def _get_itemset_cnt_iter(self, itemset, dataset):
        counter = 0
        for _, value in dataset.items():
            # Если множество кандидатов является подмножеством в текущем наборе, то увеличиваем счетчик
            if set(itemset).issubset(value):
                counter += 1
        return counter

    # подсчет количества вхождений множества-кандидата в множества из набора данных (метод apply)
    def _get_itemset_cnt_apply(self, itemset, dataset):
        return dataset.apply(lambda row: self._is_subset(itemset, row)).sum()

    # формирование списка кандидатов по порогу минимальной поддержки
    def _proc_candidates_set(self, candset, dataset, min_supp):
        print("Processing candidates...")
        df_buf = []
        # todo: можно оптимицировать цикл под объект Series (аргумент candset)
        for value in candset:
            # todo: Самая нагруженная часть кода!
            # Стандартная итерация
            supp = self._get_itemset_cnt_iter(value, dataset)
            # Метод apply
            #supp = self._get_itemset_cnt_apply(pd.Series(value), dataset)
            supp = self._get_support(supp)
            if supp > min_supp:
                df_buf.append({'Items': value, 'Support': supp})
        return pd.DataFrame(df_buf)

    # формирование начального списка кандидатов (по одному уникальному item в каждом наборе)
    # метод упрощен, т.к. не создает внутри себя отдельный Series, содержащий наборы item, т.к.
    # в качестве аргумента передается уже обработанный с помощью split объект Series (dataset)
    # todo: использовать оптимальные циклы
    def _get_items_ser2(self, itemset):
        items_list = []
        # Все наборы items из dataset добавляем в общий список
        for index, value in itemset.items():
            items_list += value
        # Конвертируем полученный список в объект Series и оставляем только уникальные значения (drop_duplicates)
        # todo: здесь можно выподнить преобразование уникальных строковых значений item в числовые
        result_ser = pd.Series(items_list, dtype=object).drop_duplicates().reset_index(drop=True)
        # Преобразуем каждый item в наборе в формат [item] (объект List) для возможности добавление новых item
        for index, value in result_ser.items():
            result_ser[index] = [value]
        return result_ser

    # Комбинируем items, которые прошли порог поддержки min_supp
    def _get_new_candidates(self, candset_old):
        print("Generating candidates...")
        candset_new = []    # список для хранения новых комбинаций items
        # в текущем списке множеств-кандидатов построчно просматриваем множетства ниже по таблице
        for index, val in candset_old.items():
            i = index + 1
            # если вышли за пределы таблицы, прерываем цикл
            if i >= candset_old.size:
                break
            # todo: здесь можно использовать рекуррентную функцию
            # иначе проверяем: содержится ли в текущем множестве (val) последний элемент из нижестоящего множества
            for _, subval in candset_old.iloc[i:].items():
                cand_new = list(val) # копируем список val
                sval = subval[-1]
                # если не содержится, то добавляем этот элемент к текущему множеству и получаем новое множество-кандидат
                if sval not in cand_new:
                    cand_new.append(subval[-1])
                    candset_new.append(cand_new)
        return pd.Series(candset_new).reset_index(drop=True)

    """ Вычисление support для всего набора входных данных
    min_supp - минимальный порог поддержки (support) """
    def get_ds_support(self, min_supp):
        # Получаем список, содержащий наборы данных из всех транзакций в виде набора списков
        items_set = self.dataset_in[self.items_col].str.split(self.items_sep)
        # Первый набор items для проверки.
        # Содержит списки, в которых находится по одному уникальному item
        candidates = self._get_items_ser2(items_set)
        print("First candidates:")
        print(candidates)
        K = 0       # Счетчик этапов генерации и обработки списков items
        while(True):
            K += 1
            # Вычисляем поддержку (support) для всех наборов items в списке кандидатов (candidates)
            # и фильтруем по пороговому значению min_supp
            validset = self._proc_candidates_set(candidates, items_set, min_supp)
            print("_____Step_%d_____" % K)
            print(validset, '\n')
            # если ни один из наборов items не прошел порог поддержки (min_supp), то завершаем цикл вычислений
            if validset.empty:
                break
            # иначе добавляем проверенное множество наборов items в итоговую таблицу данных (dataset_final)
            else:
                self.dataset_final = pd.concat([self.dataset_final, validset])
            # формируем новый список наборов-кандидатов (размером +1) из проверенного множества (validset)
            candidates = self._get_new_candidates(validset['Items'])

    def get_association_rules(self, min_confidence=30):
        """Генерация ассоциативных правил"""
        print("Generating association rules...")
        
        # Получаем исходные транзакции в виде списков
        transactions = self.dataset_in[self.items_col].str.split(self.items_sep).tolist()
        
        rules = []
        
        # Проходим по всем частым наборам
        for _, row in self.dataset_final.iterrows():
            itemset = row['Items']
            support_union = row['Support']
            
            if len(itemset) >= 2:
                from itertools import combinations
                
                for i in range(1, len(itemset)):
                    for antecedent in combinations(itemset, i):
                        consequent = tuple(set(itemset) - set(antecedent))
                        
                        support_antecedent = self._get_support_for_itemset(antecedent)
                        
                        if support_antecedent > 0:
                            confidence = (support_union / support_antecedent) * 100
                            
                            if confidence >= min_confidence:
                                support_consequent = self._get_support_for_itemset(consequent)
                                lift = (confidence / support_consequent) if support_consequent > 0 else 0

                                conviction = (1 - support_consequent/100) / (1 - confidence/100) if confidence < 100 else float('inf')
                                
                                rules.append({
                                    'Antecedent': ', '.join(antecedent),
                                    'Consequent': ', '.join(consequent),
                                    'Rule': f"{', '.join(antecedent)} → {', '.join(consequent)}",
                                    'Support': support_union,
                                    'Confidence': confidence,
                                    'Lift': lift,
                                    'Conviction': conviction
                                })
        
        self.rules_df = pd.DataFrame(rules)
        return self.rules_df
    
    def _get_support_for_itemset(self, itemset):
        """Получить поддержку для набора товаров"""
        # Ищем в dataset_final
        for _, row in self.dataset_final.iterrows():
            if set(row['Items']) == set(itemset):
                return row['Support']
        
        # Если не нашли, считаем заново
        transactions = self.dataset_in[self.items_col].str.split(self.items_sep)
        count = self._get_itemset_cnt_iter(itemset, transactions)
        return (count / len(transactions)) * 100
""" Конвертирование входных данных, имеющих вид:
T1 item1
T1 item2
T1 item3
...
в формат:
T1 item1,item2,item3,...
"""
def transform_in_dataset(dataset):
    col_id, col_item = dataset.columns.values
    ind = 0
    itemset = ""
    dataset_transformed = []
    dataset_transformed.append({ col_id: dataset.iloc[0][col_id], col_item: itemset})
    for index, row in dataset.iterrows():
        if row[col_id] == dataset_transformed[ind][col_id]:
            itemset += row[col_item] + ","
        else:
            dataset_transformed[ind][col_item] = itemset[:-1]
            ind += 1
            itemset = row[col_item] + ","
            dataset_transformed.append({ col_id: row[col_id], col_item: itemset})
    dataset_transformed[len(dataset_transformed)-1][col_item] = itemset[:-1]
    return pd.DataFrame(dataset_transformed)

apriori = ARApriori("checks.csv", sep=';', items_sep=',', items_col='ITEM')
# Трансформация данных в формат T1 item1,item2,item3,... (если необходимо)
ds_transformed = transform_in_dataset(apriori.dataset_in)
apriori.set_ds(ds_transformed)
apriori.get_ds_support(1)
# TEST
#apriori = ARApriori("test_set.csv", sep=';', items_sep=',', items_col='Items')
# Вычисляем support по заданному пороговому значению (в %)

print("_____RESULT_____")
final = apriori.dataset_final
print("Save result to CSV ?:")
q = input()
if q.lower() == 'y':
    apriori.dataset_final.to_csv('result_supp.csv', index=True, sep=";", encoding="utf-8-sig")



First candidates:
0                             [Гель для туалетов]
1                         [Сода кальцинированная]
2                [Чистящий порошок универсальный]
3                                    [Микроспрей]
4                      [Средство для чистки плит]
5                                       [Дозатор]
6                     [Стиральный порошок ручной]
7                                   [Мыло жидкое]
8                                 [Мыло кусковое]
9                                  [Отбеливатель]
10                                 [Зубная паста]
11                              [Пятновыводитель]
12                            [Салфетки бумажные]
13             [Стиральный порошок универсальный]
14                    [Средство для мытья посуды]
15                  [Средство для прочистки труб]
16                            [Перчатки тканевые]
17                             [Антистатик спрей]
18                           [Освежитель воздуха]
19                           [Пе

In [11]:
final_df = final.sort_values("Support", ascending=False)

Подсчет достоверности и лифта

In [12]:
from itertools import combinations

# создаём map support
final_df['Items_set'] = final_df['Items'].apply(set)
support_map = dict(zip(final_df['Items_set'].map(lambda x: tuple(sorted(x))), final_df['Support']))

# считаем confidence и lift
def calc_metrics(row):
    items = row['Items_set']
    
    if len(items) < 2:
        return pd.Series([None, None])
    
    best_conf = 0
    best_lift = 0
    
    for i in range(1, len(items)):
        for antecedent in combinations(items, i):
            antecedent = set(antecedent)
            consequent = items - antecedent
            
            a_key = tuple(sorted(antecedent))
            b_key = tuple(sorted(consequent))
            ab_key = tuple(sorted(items))
            
            supp_ab = support_map.get(ab_key, 0)
            supp_a = support_map.get(a_key, 0)
            supp_b = support_map.get(b_key, 0)
            
            if supp_a > 0 and supp_b > 0:
                conf = supp_ab / supp_a
                lift = conf / (supp_b / 100)
                
                if conf > best_conf:
                    best_conf = conf
                    best_lift = lift
    
    return pd.Series([best_conf * 100, best_lift])
final_df.drop(columns=["Items_set"])
final_df[['Confidence', 'Lift']] = final_df.apply(calc_metrics, axis=1)

In [13]:
final_df['Items_temp'] = final_df['Items'].apply(lambda x: tuple(sorted(x)))
final_df = final_df.drop_duplicates(subset='Items_temp')
final_df = final_df.drop(columns='Items_temp')

Топ-10 значений по поддержке среди наборов с 2 и более товарами:

In [14]:
final_df[final_df["Items"].str.len() > 1].sort_values("Support", ascending=False)[:10]

,Items,Support,Items_set,Confidence,Lift
36,"[Мыло жидкое, Мыло кусковое]",8.166259,"{Мыло жидкое, Мыло кусковое}",83.500000,4.356059
10,"[Чистящий порошок универсальный, Микроспрей]",6.405868,"{Микроспрей, Чистящий порошок универсальный}",44.863014,1.671127
40,"[Мыло кусковое, Средство для мытья посуды]",6.405868,"{Мыло кусковое, Средство для мытья посуды}",86.184211,4.496090
7,"[Сода кальцинированная, Чистящий порошок униве...",4.694377,"{Сода кальцинированная, Чистящий порошок униве...",72.180451,5.055104
19,"[Микроспрей, Мыло кусковое]",4.449878,"{Микроспрей, Мыло кусковое}",23.214286,0.864722
64,"[Кондиционер для белья, Стиральный порошок-авт...",3.863081,"{Кондиционер для белья, Стиральный порошок-авт...",84.946237,12.063545
12,"[Чистящий порошок универсальный, Зубная паста]",3.765281,"{Зубная паста, Чистящий порошок универсальный}",26.736111,1.872443
14,"[Чистящий порошок универсальный, Средство от н...",3.716381,"{Средство от накипи, Чистящий порошок универса...",69.724771,4.883122
23,"[Микроспрей, Освежитель воздуха]",3.227384,"{Микроспрей, Освежитель воздуха}",31.730769,1.181957
4,"[Гель для туалетов, Мыло жидкое]",3.227384,"{Мыло жидкое, Гель для туалетов}",33.000000,3.053620


Топ-10 по лифту

In [19]:
final_df[final_df["Items"].str.len() > 1].sort_values("Lift", ascending=False)[:10]

,Items,Support,Items_set,Confidence,Lift
0,"[Гель для туалетов, Сода кальцинированная, Чис...",1.662592,"{Сода кальцинированная, Гель для туалетов, Чис...",100.000000,15.375940
28,"[Микроспрей, Кондиционер для белья, Стиральный...",1.026895,"{Кондиционер для белья, Микроспрей, Стиральный...",100.000000,14.201389
48,"[Мыло кусковое, Антистатик спрей, Средство для...",1.369193,"{Антистатик спрей, Мыло кусковое, Средство для...",100.000000,13.453947
33,"[Средство для чистки плит, Мыло кусковое, Сред...",1.809291,"{Мыло кусковое, Средство для чистки плит, Сред...",100.000000,13.453947
44,"[Мыло кусковое, Средство для мытья посуды, Пен...",1.124694,"{Пена/соль для ванн, Мыло кусковое, Средство д...",100.000000,13.453947
64,"[Кондиционер для белья, Стиральный порошок-авт...",3.863081,"{Кондиционер для белья, Стиральный порошок-авт...",84.946237,12.063545
4,"[Гель для туалетов, Мыло жидкое, Мыло кусковое]",2.738386,"{Мыло жидкое, Гель для туалетов, Мыло кусковое}",100.000000,10.225000
38,"[Стиральный порошок ручной, Мыло жидкое, Мыло ...",1.418093,"{Мыло жидкое, Стиральный порошок ручной, Мыло ...",100.000000,10.225000
62,"[Пена/соль для ванн, Запасной баллон для освеж...",1.369193,"{Пена/соль для ванн, Запасной баллон для освеж...",31.818182,7.311032
14,"[Чистящий порошок универсальный, Микроспрей, С...",1.760391,"{Микроспрей, Чистящий порошок универсальный, С...",100.000000,7.003425


Топ-10 значений по поддержке среди наборов, состоящих из 2 товаров:

In [16]:
final_df[final_df["Items"].str.len() == 2].sort_values("Support", ascending=False)[:10]

,Items,Support,Items_set,Confidence,Lift
36,"[Мыло жидкое, Мыло кусковое]",8.166259,"{Мыло жидкое, Мыло кусковое}",83.500000,4.356059
10,"[Чистящий порошок универсальный, Микроспрей]",6.405868,"{Микроспрей, Чистящий порошок универсальный}",44.863014,1.671127
40,"[Мыло кусковое, Средство для мытья посуды]",6.405868,"{Мыло кусковое, Средство для мытья посуды}",86.184211,4.496090
7,"[Сода кальцинированная, Чистящий порошок униве...",4.694377,"{Сода кальцинированная, Чистящий порошок униве...",72.180451,5.055104
19,"[Микроспрей, Мыло кусковое]",4.449878,"{Микроспрей, Мыло кусковое}",23.214286,0.864722
64,"[Кондиционер для белья, Стиральный порошок-авт...",3.863081,"{Кондиционер для белья, Стиральный порошок-авт...",84.946237,12.063545
12,"[Чистящий порошок универсальный, Зубная паста]",3.765281,"{Зубная паста, Чистящий порошок универсальный}",26.736111,1.872443
14,"[Чистящий порошок универсальный, Средство от н...",3.716381,"{Средство от накипи, Чистящий порошок универса...",69.724771,4.883122
23,"[Микроспрей, Освежитель воздуха]",3.227384,"{Микроспрей, Освежитель воздуха}",31.730769,1.181957
4,"[Гель для туалетов, Мыло жидкое]",3.227384,"{Мыло жидкое, Гель для туалетов}",33.000000,3.053620


Тест на влияние критического значения поддержки на время исполнения алгоритма:

In [17]:
import datetime

min_sups = range(1, 10)
times = []

for min_sup in min_sups:
    start_time = datetime.datetime.now()
    apriori.get_ds_support(min_sup)
    times.append((datetime.datetime.now() - start_time).seconds)


First candidates:
0                             [Гель для туалетов]
1                         [Сода кальцинированная]
2                [Чистящий порошок универсальный]
3                                    [Микроспрей]
4                      [Средство для чистки плит]
5                                       [Дозатор]
6                     [Стиральный порошок ручной]
7                                   [Мыло жидкое]
8                                 [Мыло кусковое]
9                                  [Отбеливатель]
10                                 [Зубная паста]
11                              [Пятновыводитель]
12                            [Салфетки бумажные]
13             [Стиральный порошок универсальный]
14                    [Средство для мытья посуды]
15                  [Средство для прочистки труб]
16                            [Перчатки тканевые]
17                             [Антистатик спрей]
18                           [Освежитель воздуха]
19                           [Пе

Проверка времени исполнения алгоритма при различных пороговых уровнях поддержки:

In [18]:
for sup, time in zip(min_sups, times):
    print(f"Пороговое значение support: {sup}, время обработки: {time} с.")

Пороговое значение support: 1, время обработки: 12 с.
Пороговое значение support: 2, время обработки: 3 с.
Пороговое значение support: 3, время обработки: 1 с.
Пороговое значение support: 4, время обработки: 1 с.
Пороговое значение support: 5, время обработки: 0 с.
Пороговое значение support: 6, время обработки: 0 с.
Пороговое значение support: 7, время обработки: 0 с.
Пороговое значение support: 8, время обработки: 0 с.
Пороговое значение support: 9, время обработки: 0 с.


Чем ниже критическое значение поддержки, тем дольше работает алгоритм

#### Генерация правил

In [20]:
def generate_rules(final_df, min_conf=30):
    rules = []
    
    support_map = dict(zip(
        final_df['Items'].apply(lambda x: tuple(sorted(x))),
        final_df['Support']
    ))
    
    for _, row in final_df.iterrows():
        items = set(row['Items'])
        
        if len(items) < 2:
            continue
        
        for i in range(1, len(items)):
            for antecedent in combinations(items, i):
                antecedent = set(antecedent)
                consequent = items - antecedent
                
                a_key = tuple(sorted(antecedent))
                b_key = tuple(sorted(consequent))
                ab_key = tuple(sorted(items))
                
                supp_ab = support_map.get(ab_key, 0)
                supp_a = support_map.get(a_key, 0)
                supp_b = support_map.get(b_key, 0)
                
                if supp_a == 0:
                    continue
                
                confidence = supp_ab / supp_a * 100
                
                if confidence >= min_conf:
                    lift = (confidence / 100) / (supp_b / 100) if supp_b > 0 else 0
                    
                    rules.append({
                        'Rule': f"{', '.join(antecedent)} → {', '.join(consequent)}",
                        'Support': supp_ab,
                        'Confidence': confidence,
                        'Lift': lift
                    })
    
    return pd.DataFrame(rules)

In [27]:
rules_df = generate_rules(final_df, min_conf=30)

rules_df.sort_values(by='Lift', ascending=False).head(15)["Rule"].to_list()

['Гель для туалетов, Чистящий порошок универсальный → Сода кальцинированная',
 'Кондиционер для белья, Микроспрей → Стиральный порошок-автомат',
 'Мыло кусковое, Средство для чистки плит → Средство для мытья посуды',
 'Пена/соль для ванн, Мыло кусковое → Средство для мытья посуды',
 'Антистатик спрей, Мыло кусковое → Средство для мытья посуды',
 'Стиральный порошок-автомат → Кондиционер для белья',
 'Кондиционер для белья → Стиральный порошок-автомат',
 'Микроспрей, Стиральный порошок-автомат → Кондиционер для белья',
 'Гель для туалетов, Мыло кусковое → Мыло жидкое',
 'Стиральный порошок ручной, Мыло кусковое → Мыло жидкое',
 'Пена/соль для ванн → Запасной баллон для освежителя',
 'Запасной баллон для освежителя → Пена/соль для ванн',
 'Микроспрей, Средство от накипи → Чистящий порошок универсальный',
 'Сода кальцинированная, Микроспрей → Чистящий порошок универсальный',
 'Бумажное полотенце → Освежитель воздуха']

#### Выводы:

- Среди приведенных покупок, чаще всего вместе берут различные средства гигиены:

    разные виды мыла (Мыло жидкое -- Мыло кусковое);

    средства для стирки (Кондиционер для белья -- Стиральный порошок-автомат);

    средства для мытья посуды (Чистящий порошок универсальный -- Средство от накипи, Сода кальцинированная -- Чистящий порошок универсальный и пр.)

- Можно сделать предположение о частых совместных покупках товаров, находящихся рядом на полке / в одном разделе в магазине:

    Микроспрей -- Освежитель воздуха

- Ассоциативные правила не позволяют автоматически выделить товары-субституты (взаимозаменяемые товары), так как значения поддержки недостаточно для различия потенциальных товаров-субститутов от товаров-комплементов -- среди наборов с высокой поддержкой встречаются как субституты (жидкое и кусковое мыло в одном наборе с гелем для туалетов), так и несвязные с ними товары (стиральный порошок ручной в одном наборе с гелем для туалетов)